# 05 Model comparison, Figs 3-4 and Extended Data Fig. 3

Reads the per-lane curves and summaries in `data/per_lane`, and writes to `MFDs_FINAL`. The constants a, d, s_u, s_K and those of the urbanization index, with its standardization moments, come from `MFDs_Curve_uncertainty/point_estimate.json`, the single fit used everywhere. The standalone cubic keeps a = 0.157 (`scripts/cubic_factors.py`). Map averages come from `data/map_metrics.csv`; they carry zero weight in the index.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "scripts")))
from paths import MAP_METRICS, MFD_DIR, OUTPUTS_DIR, RESULTS

# run for results of different models 

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1) Paths (edit if needed)
# ----------------------------
base_mfds = MFD_DIR
data_pattern = os.path.join(base_mfds, "data_*_grouped.csv")

outputs_dir = OUTPUTS_DIR

# where to save figures and tables
results_dir = os.path.join(RESULTS, "MFDs_FINAL")

os.makedirs(results_dir, exist_ok=True)

figs_dir = os.path.join(results_dir, "figures_per_city")
os.makedirs(figs_dir, exist_ok=True)

# NEW: figures without Hybrid
figs_dir_nohyb = os.path.join(results_dir, "figures_per_city_no_hybrid")
os.makedirs(figs_dir_nohyb, exist_ok=True)

# ----------------------------
# 2) Constants
# ----------------------------
EPS = 1e-9

# number of K/Kc bins on [0, 1.1] for bin-based errors
NBINS = 6
BIN_EDGES = np.linspace(0.0, 1.1, NBINS + 1)
BIN_CENTERS = 0.5 * (BIN_EDGES[:-1] + BIN_EDGES[1:])
# ---- Calibrated hybrid parameters ----
# every fitted constant comes from the single fit on the corrected data
# (mfd_uncertainty.py point estimate), instead of being typed in.
import json
PE_PATH = os.path.join(RESULTS, "MFDs_Curve_uncertainty", "point_estimate.json")
_pe = json.load(open(PE_PATH))
_hp = _pe["hybrid_params"]
U_TH = _pe["fixed"]["u_th"]   # threshold (urbanization) for alpha
S_U  = _hp["s_u"]              # alpha sharpness
S_K  = _hp["s_K"]              # beta sharpness around kc
alph = _hp["a"]                # a in the cubic part of the hybrid equation
alp  = 0.157                   # a in the pure cubic equation (standalone; unchanged)
EXP_A0 = _hp["d"]              # d, amplitude of the exponential branch
EXP_A1 = _pe["fixed"]["a1_EXP_A1"]
# Urbanization Index (eq. 6), metrics in order [uf, avg_link_length, avg_speed_limit,
# node_connectivity]; moments are the fit's own sample moments over the 141 networks.
_mom = _pe["ui_moments"]
METRIC_MEAN = np.array(_mom["metric_mean"], float)
METRIC_STD  = np.array(_mom["metric_std"], float)
PHI_MEAN = np.array(_mom["phi_mean"], float)
PHI_STD  = np.array(_mom["phi_std"], float)
b_ui = _hp["b"]
w_ui = np.array([_hp["w_uf"], _hp["w_L"], _hp["w_SL"], _hp["w_C"]], float)
print(f"[CONSTANTS] a={alph:.4f} d={EXP_A0:.4f} s_u={S_U} s_K={S_K} b={b_ui:.4f} w={w_ui}")

_map_metrics = pd.read_csv(MAP_METRICS).set_index("city")

def get_city_network_metrics(city):
    """Return (avg_link_length, avg_speed_limit, node_connectivity) for this city, from data/map_metrics.csv."""
    if city not in _map_metrics.index:
        print(f"[WARN] no map metrics for '{city}'.")
        return None
    r = _map_metrics.loc[city]
    return float(r.avg_link_length), float(r.avg_speed_limit), float(r.node_connectivity)

def compute_ui_from_metrics(uf, avg_link_length, avg_speed_limit, node_connectivity):
    """
    Compute Urbanization Index I for a single city given:
      uf, avg_link_length, avg_speed_limit, node_connectivity
    using the exp-form equation learned from joint calibration.
    """
    # shape (1,4) to reuse vectorized formula
    M = np.array([[uf, avg_link_length, avg_speed_limit, node_connectivity]], float)

    Z0 = (M - METRIC_MEAN) / METRIC_STD
    Z0 = np.clip(Z0, -2.0, 2.0)

    Phi = np.exp(Z0)              # (1,4)
    Zphi = (Phi - PHI_MEAN) / PHI_STD

    I = b_ui + Zphi @ w_ui        # (1,)
    return float(I[0])

# ----------------------------
# 3) Utilities
# ----------------------------
def read_outputs_for_city(city):
    """Read outputs_{city}.txt -> (qc, kc, q_kc_6, q_kc_3, q_kc_2, q_2kc_3, q_5kc_6)"""
    path = os.path.join(outputs_dir, f"outputs_{city}.txt")
    if not os.path.exists(path):
        return None
    try:
        parts = [p.strip() for p in open(path, "r").read().strip().split(",")]
        if len(parts) != 7:
            return None
        return tuple(map(float, parts))
    except Exception:
        return None

def nonagonal_points_core(kc, uf):
    # Only up to Kc; tail will be added separately
    return [
        (0.0, 0.0),
        (kc/6.0, kc*uf/6.0),
        (kc/3.0, kc*uf*0.971/3.0),
        (kc/2.0, kc*uf*0.934/2.0),
        (2.0*kc/3.0, 2.0*kc*uf*0.878/3.0),
        (5.0*kc/6.0, 5.0*kc*uf*0.798/6.0),
        (kc, kc*uf*0.719),
    ]

def pentagonal_points_core(kc, uf):
    # Only up to Kc; tail will be added separately
    return [
        (0.0, 0.0),
        (kc/2.0, kc*uf*0.934/2.0),
        (kc, kc*uf*0.719),
    ]

def concave_tail_after_kc(kc, q_at_kc, theta=0.6):
    """
    Build a concave two-segment tail from (Kc, Qc) to (1.8Kc, 0)
    via (1.4Kc, theta*Qc), with theta>0.5 ensuring the second segment
    is steeper (more negative slope) than the first.
    """
    k1 = 1.4 * kc
    k2 = 1.8 * kc
    q1 = theta * q_at_kc
    return [(kc, q_at_kc), (k1, q1), (k2, 0.0)]

def build_piecewise_xy(points):
    k_pts = np.array([p[0] for p in points], float)
    q_pts = np.array([p[1] for p in points], float)
    ks, qs = [], []
    total_span = max(EPS, k_pts[-1] - k_pts[0])
    for i in range(len(k_pts)-1):
        k0, k1 = k_pts[i], k_pts[i+1]
        q0, q1 = q_pts[i], q_pts[i+1]
        nseg = max(2, int(abs(k1 - k0) / total_span * 300))
        k_lin = np.linspace(k0, k1, nseg, endpoint=(i == len(k_pts)-2))
        q_lin = np.linspace(q0, q1, nseg, endpoint=(i == len(k_pts)-2))
        ks.append(k_lin); qs.append(q_lin)
    return np.concatenate(ks), np.concatenate(qs)

def q_greenshields(k, uf, kj):
    return uf * k * (1.0 - k / kj)

def q_underwood(k, uf, kc):
    return uf * k * np.exp(-k / kc)

def q_exp_generic(k, uf, kc, a0=EXP_A0, a1=EXP_A1):
    """
    Generic exponential model for the Hybrid's low-urbanization branch:
        q_exp(k) = a0 * uf * k * exp( -a1 * k / kc )

    With a0 = 1, a1 = 1 this is exactly the Underwood model.
    """
    k = np.asarray(k, float)
    return a0 * uf * k * np.exp(-a1 * k / kc)

def q_drake(k, uf, kc):
    return uf * k * np.exp(-0.5 * (k / kc)**2)

def q_cubic(k, uf, kc, a):
    # q = uf*k + a*uf*k^2/kc - ((1+2a)/3)*uf*k^3/kc^2
    k = np.asarray(k, float)
    return uf * (k + a*(k**2)/kc - (2*a+1)*(k**3)/(3*kc**2))

# ---- decreasing logistic used for both alpha(uf) and beta(k) ----
def dec_sigmoid(z):
    z = np.clip(z, -60.0, 60.0)
    return 1.0 / (1.0 + np.exp(z))

def alpha_from_uf(u_index, u_th=U_TH, s_u=S_U):
    """
    Alpha as a decreasing logistic in (UI - U_TH)/S_U, where UI is the
    Urbanization Index for the city.

    Lower UI (<< U_TH)  -> alpha ~ 1  -> larger gamma -> Underwood dominates.
    Higher UI (>> U_TH) -> alpha ~ 0  -> gamma small  -> Cubic dominates.
    """
    z = (u_index - u_th) / max(EPS, s_u)
    return dec_sigmoid(z)

def beta_from_k(k, kc, s_k=S_K):
    x_minus_1 = (np.asarray(k, float) / max(EPS, kc)) - 1.0
    return dec_sigmoid(x_minus_1 / max(EPS, s_k))

def q_hybrid(k, uf, u_index, kc, u_th=U_TH, s_u=S_U, s_k=S_K, a_h=alph):
    """
    gamma = alpha(UI) * beta(k)
    q_cubChoice = alpha*q_cubT + (1-alpha)*q_cub
    q = gamma*q_exp + (1-gamma)*q_cubChoice

    where:
      - q_exp is a generic exponential: q_exp = EXP_A0 * uf * k * exp( -EXP_A1 * k / kc )
      - q_cub is the cubic branch with coefficient a_h
      - q_cubT is a scaled cubic tail such that q_cubT(Kc) = q_exp(Kc)

    IMPORTANT:
      - uf      is still used inside the exponential and cubic flows.
      - u_index is the Urbanization Index, used only in alpha().
      - With EXP_A0 = 1, EXP_A1 = 1, the exponential branch is exactly Underwood.
    """
    # Generic exponential branch (low-urbanization side)
    q_exp = q_exp_generic(k, uf, kc, a0=EXP_A0, a1=EXP_A1)

    # Cubic branch
    q_cub = q_cubic(k, uf, kc, a_h)

    # Compute tail scaling so that q_cubT(Kc) matches q_exp(Kc)
    q_exp_kc = q_exp_generic(kc, uf, kc, a0=EXP_A0, a1=EXP_A1)
    q_cub_kc = q_cubic(kc, uf, kc, a_h)
    denom = q_cub_kc if abs(q_cub_kc) > EPS else EPS
    scale_tail = q_exp_kc / denom
    q_cubT = scale_tail * q_cub

    # Blend coefficients
    alpha = alpha_from_uf(u_index, u_th, s_u)   # scalar per city, from UI
    beta  = beta_from_k(k, kc, s_k)             # array over k
    gamma = alpha * beta

    q_cubChoice = alpha * q_cubT + (1.0 - alpha) * q_cub
    return gamma * q_exp + (1.0 - gamma) * q_cubChoice

# ---- rightmost zero of the cubic (in k) ----
def cubic_right_zero_k(kc, a):
    b = -(1.0 + 2.0*a) / 3.0
    D = a*a - 4.0*b
    if D < 0:
        return float('nan')
    sqrtD = math.sqrt(D)
    denom = 2.0 * b
    roots = []
    if denom != 0.0:
        x1 = (-a + sqrtD) / denom
        x2 = (-a - sqrtD) / denom
        for x in (x1, x2):
            if np.isfinite(x) and x > 0:
                roots.append(x)
    if not roots:
        return float('nan')
    x_star = max(roots)
    return float(x_star * kc)

def interpolate_piecewise(points, k_eval):
    pts = sorted(points, key=lambda x: x[0])
    k_pts = np.array([p[0] for p in pts], float)
    q_pts = np.array([p[1] for p in pts], float)
    return np.interp(k_eval, k_pts, q_pts, left=q_pts[0], right=q_pts[-1])

def normalized_errors(q_obs, q_mod, q_scale):
    if not np.isfinite(q_scale) or q_scale <= 0:
        q_scale = np.nanmax(q_obs) if np.nanmax(q_obs) > 0 else 1.0
    qo = np.asarray(q_obs, float) / q_scale
    qm = np.asarray(q_mod, float) / q_scale
    diff = qo - qm
    rmse = np.sqrt(np.nanmean(diff**2))
    mae  = np.nanmean(np.abs(diff))
    return rmse, mae

# ----------------------------
# 4) Main loop
# ----------------------------
rows = []
curve_names = ["Pentagonal", "Nonagonal", "Cubic", "Greenshields", "Underwood", "Drake", "Hybrid"]
curves_no_hybrid = ["Pentagonal", "Nonagonal", "Cubic", "Greenshields", "Underwood", "Drake"]

# Structures for bin-based errors: list of per-city-bin RMSE/MAE per curve and bin
bin_rmse_lists = {c: [ [] for _ in range(NBINS) ] for c in curve_names}
bin_mae_lists  = {c: [ [] for _ in range(NBINS) ] for c in curve_names}

csv_files = sorted(glob.glob(data_pattern))
if not csv_files:
    print("No data_*_grouped.csv files found at:", data_pattern)

for data_fp in csv_files:
    cityfile = os.path.basename(data_fp)
    city_name = (
        cityfile[5:-12]
        if (cityfile.lower().startswith("data_") and cityfile.lower().endswith("_grouped.csv"))
        else cityfile.replace("data_", "").replace("_grouped.csv", "")
    )

    outs = read_outputs_for_city(city_name)
    if outs is None:
        print(f"[WARN] Missing or bad outputs for city '{city_name}'. Skipping.")
        continue

    qc, kc_raw, q_kc6, q_kc3, q_kc2, q_2kc3, q_5kc6 = outs

    # uf & kc guards
    if not (kc_raw and np.isfinite(kc_raw) and kc_raw > 0):
        print(f"[WARN] kc invalid for '{city_name}' (kc_raw={kc_raw}). Skipping.")
        continue

    uf = 6.0 * q_kc6 / kc_raw
    if not (np.isfinite(uf) and uf > 0):
        print(f"[WARN] uf invalid for '{city_name}' (uf={uf}). Skipping.")
        continue

    # --- NEW: compute Urbanization Index for this city from metrics (fixed exp form) ---
    net_metrics = get_city_network_metrics(city_name)
    if net_metrics is None:
        print(f"[WARN] Network metrics missing/invalid for '{city_name}'. Skipping.")
        continue

    avg_link_length, avg_speed_limit, node_connectivity = net_metrics

    u_index = compute_ui_from_metrics(
        uf=uf,
        avg_link_length=avg_link_length,
        avg_speed_limit=avg_speed_limit,
        node_connectivity=node_connectivity,
    )
    if not np.isfinite(u_index):
        print(f"[WARN] Urbanization index NaN/inf for '{city_name}' (UI={u_index}). Skipping.")
        continue

    # Use kc from outputs for all models
    kc = kc_raw
    if not np.isfinite(kc) or kc <= 0:
        print(f"[WARN] kc invalid for '{city_name}' (kc={kc}). Skipping.")
        continue

    # Critical point for scaling and plotting
    kce = kc       # treat "kce" as Kc
    qce = qc       # use qc from the outputs file

    # load & clean data
    try:
        df = pd.read_csv(data_fp)
    except Exception as e:
        print(f"[WARN] Cannot read {data_fp}: {e}")
        continue
    if "density" not in df.columns or "flow" not in df.columns:
        print(f"[WARN] 'density'/'flow' columns not found in {data_fp}. Skipping.")
        continue

    dens = pd.to_numeric(df["density"], errors="coerce")
    flow = pd.to_numeric(df["flow"], errors="coerce")
    mask = (
        dens.ge(0) & dens.le(1.1 * kc) &
        flow.ge(0) &
        dens.replace([np.inf, -np.inf], np.nan).notna() &
        flow.replace([np.inf, -np.inf], np.nan).notna()
    )
    df_clean = pd.DataFrame({"density": dens[mask], "flow": flow[mask]}).dropna()
    if df_clean.empty:
        print(f"[WARN] No valid rows after cleaning for '{city_name}'. Skipping.")
        continue

    k_obs = df_clean["density"].to_numpy(float)
    q_obs = df_clean["flow"].to_numpy(float)

    # ------------- Build curves -------------
    # core (up to Kc)
    non_core  = nonagonal_points_core(kc, uf)
    pent_core = pentagonal_points_core(kc, uf)

    # concave two-segment tails assuming Kj = 1.8*Kc
    non_tail  = concave_tail_after_kc(kc, non_core[-1][1],  theta=0.6)
    pent_tail = concave_tail_after_kc(kc, pent_core[-1][1], theta=0.6)

    # full piecewise sequences
    non_piece = non_core + non_tail[1:]   # avoid duplicating (kc, qc)
    pent_pts  = pent_core + pent_tail[1:]

    k_non,  q_non  = build_piecewise_xy(non_piece)
    k_pent, q_pent = build_piecewise_xy(pent_pts)

    # --- compute zero-crossings for plot range (later of cubic & hybrid), add 15% padding ---
    k0_cub  = cubic_right_zero_k(kc, alp)   # standalone cubic
    k0_hyb  = cubic_right_zero_k(kc, alph)  # hybrid's cubic
    k0_late = k0_cub if (np.isfinite(k0_cub) and
                         (not np.isfinite(k0_hyb) or k0_cub >= k0_hyb)) else k0_hyb
    if not np.isfinite(k0_late) or k0_late <= 0:
        k0_late = 1.8 * kc  # at least show to our assumed Kj
    k_plot_max = 1.15 * k0_late  # 15% padding

    # Build grid wide enough to include the padded zero point
    k_grid = np.linspace(0.0, k_plot_max, 900)

    # For models that require Kj (e.g., Greenshields), use Kj = 2*Kc
    kj_model = 2.0 * kc

    q_cub   = q_cubic(k_grid, uf, kc, alp)          # standalone cubic with 'alp'
    q_green = q_greenshields(k_grid, uf, kj_model)
    q_under = q_underwood(k_grid, uf, kc)
    q_drak  = q_drake(k_grid, uf, kc)
    q_hyb   = q_hybrid(k_grid, uf, u_index, kc, u_th=U_TH, s_u=S_U, s_k=S_K, a_h=alph)
    alpha_uf = float(alpha_from_uf(u_index, U_TH, S_U))  # really alpha(UI), kept name for continuity

    # ------------- Errors (normalized by qce) -------------
    q_non_at_obs   = interpolate_piecewise(non_piece, k_obs)
    q_pent_at_obs  = interpolate_piecewise(pent_pts,  k_obs)
    q_cub_at_obs   = q_cubic(k_obs,   uf, kc, alp)
    q_green_at_obs = q_greenshields(k_obs, uf, kj_model)
    q_under_at_obs = q_underwood(k_obs, uf, kc)
    q_drak_at_obs  = q_drake(k_obs,   uf, kc)
    q_hyb_at_obs   = q_hybrid(k_obs,  uf, u_index, kc, u_th=U_TH, s_u=S_U, s_k=S_K, a_h=alph)

    qscale = qce
    rmse_pent, mae_pent = normalized_errors(q_obs, q_pent_at_obs,  qscale)
    rmse_non,  mae_non  = normalized_errors(q_obs, q_non_at_obs,   qscale)
    rmse_cub,  mae_cub  = normalized_errors(q_obs, q_cub_at_obs,   qscale)
    rmse_gre,  mae_gre  = normalized_errors(q_obs, q_green_at_obs, qscale)
    rmse_und,  mae_und  = normalized_errors(q_obs, q_under_at_obs, qscale)
    rmse_dra,  mae_dra  = normalized_errors(q_obs, q_drak_at_obs,  qscale)
    rmse_hyb,  mae_hyb  = normalized_errors(q_obs, q_hyb_at_obs,   qscale)

    rows.append({
        "city": city_name,
        "uf": uf, "kce": kce, "qce": qce,
        "alpha_uf": alpha_uf,
        "RMSE_Pentagonal": rmse_pent, "MAE_Pentagonal": mae_pent,
        "RMSE_Nonagonal":  rmse_non,  "MAE_Nonagonal":  mae_non,
        "RMSE_Cubic":      rmse_cub,  "MAE_Cubic":      mae_cub,
        "RMSE_Greenshields": rmse_gre, "MAE_Greenshields": mae_gre,
        "RMSE_Underwood":    rmse_und, "MAE_Underwood":    mae_und,
        "RMSE_Drake":        rmse_dra, "MAE_Drake":        mae_dra,
        "RMSE_Hybrid":       rmse_hyb, "MAE_Hybrid":       mae_hyb,
    })

    # ---------- BIN-BASED ERRORS PER CITY ----------
    k_ratio = k_obs / kc  # K/Kc in [0, ~1.1]

    model_dict = {
        "Pentagonal":   q_pent_at_obs,
        "Nonagonal":    q_non_at_obs,
        "Cubic":        q_cub_at_obs,
        "Greenshields": q_green_at_obs,
        "Underwood":    q_under_at_obs,
        "Drake":        q_drak_at_obs,
        "Hybrid":       q_hyb_at_obs,
    }

    for curve, q_mod_vec in model_dict.items():
        for b in range(NBINS):
            lo = BIN_EDGES[b]
            hi = BIN_EDGES[b + 1]
            if b < NBINS - 1:
                mask_bin = (k_ratio >= lo) & (k_ratio < hi)
            else:
                # include right edge in last bin
                mask_bin = (k_ratio >= lo) & (k_ratio <= hi + 1e-9)

            if np.any(mask_bin):
                rmse_bin, mae_bin = normalized_errors(
                    q_obs[mask_bin],
                    q_mod_vec[mask_bin],
                    qscale
                )
                bin_rmse_lists[curve][b].append(rmse_bin)
                bin_mae_lists[curve][b].append(mae_bin)

    # ------------- Plot per-city figure (WITH Hybrid) -------------
    plt.figure(figsize=(9, 6), dpi=140)

    # scatter to the background
    plt.scatter(k_obs, q_obs, s=6, alpha=0.15, c='gray', zorder=1, label="Data (scatter)")

    # curves in the requested legend order (+ Hybrid)
    h1, = plt.plot(k_pent, q_pent, linewidth=2.0, zorder=3, label="Pentagonal")
    h2, = plt.plot(k_non,  q_non,  linewidth=2.0, zorder=3, label="Nonagonal")
    h3, = plt.plot(k_grid, q_cub,  linewidth=2.0, zorder=3, label="Cubic")
    h4, = plt.plot(k_grid, q_green,linewidth=2.0, zorder=3, label="Greenshields")
    h5, = plt.plot(k_grid, q_under,linewidth=2.0, zorder=3, label="Underwood")
    h6, = plt.plot(k_grid, q_drak, linewidth=2.0, zorder=3, label="Drake")
    h7, = plt.plot(k_grid, q_hyb,  linewidth=2.6, zorder=4, label=f"Hybrid (α={alpha_uf:.2f})")

    if np.isfinite(kce) and np.isfinite(qce):
        plt.scatter([kce], [qce], s=30, zorder=4)

    plt.title(
        f"MFD fits: {city_name}\n"
        f"(uf={uf:.3f} km/h, kce={kce if np.isfinite(kce) else float('nan'):.3f}, "
        f"qce={qce if np.isfinite(qce) else float('nan'):.3f})"
    )
    plt.xlabel("Density k (veh/km/lane)")
    plt.ylabel("Flow q (veh/h/lane)")

    # --- x-limits: to padded later-zero ---
    plt.xlim(0, k_plot_max)

    ymax = np.nanmax([
        np.nanmax(q_obs) if len(q_obs) else 0,
        q_pent.max(), q_non.max(), q_cub.max(),
        q_green.max(), q_under.max(), q_drak.max(), q_hyb.max()
    ])
    plt.ylim(0, max(1.0, 1.05 * ymax))

    handles = [h1, h2, h3, h4, h5, h6, h7]
    labels  = [h.get_label() for h in handles]
    plt.legend(handles, labels, loc="upper right", fontsize=8, ncol=2)
    plt.grid(alpha=0.15)

    out_png = os.path.join(figs_dir, f"{city_name}_mfd_fits.png")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

    # ------------- NEW: Plot per-city figure (WITHOUT Hybrid) -------------
    plt.figure(figsize=(9, 6), dpi=140)

    # scatter to the background
    plt.scatter(k_obs, q_obs, s=6, alpha=0.15, c='gray', zorder=1, label="Data (scatter)")

    # same curves, just no Hybrid
    h1, = plt.plot(k_pent, q_pent, linewidth=2.0, zorder=3, label="Pentagonal")
    h2, = plt.plot(k_non,  q_non,  linewidth=2.0, zorder=3, label="Nonagonal")
    h3, = plt.plot(k_grid, q_cub,  linewidth=2.0, zorder=3, label="Cubic")
    h4, = plt.plot(k_grid, q_green,linewidth=2.0, zorder=3, label="Greenshields")
    h5, = plt.plot(k_grid, q_under,linewidth=2.0, zorder=3, label="Underwood")
    h6, = plt.plot(k_grid, q_drak, linewidth=2.0, zorder=3, label="Drake")

    if np.isfinite(kce) and np.isfinite(qce):
        plt.scatter([kce], [qce], s=30, zorder=4)

    plt.title(
        f"MFD fits (no Hybrid): {city_name}\n"
        f"(uf={uf:.3f} km/h, kce={kce if np.isfinite(kce) else float('nan'):.3f}, "
        f"qce={qce if np.isfinite(qce) else float('nan'):.3f})"
    )
    plt.xlabel("Density k (veh/km/lane)")
    plt.ylabel("Flow q (veh/h/lane)")

    plt.xlim(0, k_plot_max)

    ymax_nohyb = np.nanmax([
        np.nanmax(q_obs) if len(q_obs) else 0,
        q_pent.max(), q_non.max(), q_cub.max(),
        q_green.max(), q_under.max(), q_drak.max()
    ])
    plt.ylim(0, max(1.0, 1.05 * ymax_nohyb))

    handles_nohyb = [h1, h2, h3, h4, h5, h6]
    labels_nohyb  = [h.get_label() for h in handles_nohyb]
    plt.legend(handles_nohyb, labels_nohyb, loc="upper right", fontsize=8, ncol=2)
    plt.grid(alpha=0.15)

    out_png_nohyb = os.path.join(figs_dir_nohyb, f"{city_name}_mfd_fits_nohybrid.png")
    plt.tight_layout()
    plt.savefig(out_png_nohyb)
    plt.close()

# ----------------------------
# 5) Save errors table + summary plots (overall)
# ----------------------------
errors_df = pd.DataFrame(rows)
errors_csv = os.path.join(results_dir, "per_city_errors_normalized.csv")
errors_df.to_csv(errors_csv, index=False)

rmse_cols = [c for c in errors_df.columns if c.startswith("RMSE_")]
mae_cols  = [c for c in errors_df.columns if c.startswith("MAE_")]

rmse_mean = errors_df[rmse_cols].mean(numeric_only=True)
mae_mean  = errors_df[mae_cols].mean(numeric_only=True)

summary_rows = []
for curve in curve_names:
    summary_rows.append({
        "Curve": curve,
        "RMSE_mean": rmse_mean.get(f"RMSE_{curve}", np.nan),
        "MAE_mean":  mae_mean.get(f"MAE_{curve}", np.nan),
    })
summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("MAE_mean", ascending=True)
    .reset_index(drop=True)
)

summary_csv = os.path.join(results_dir, "errors_summary_across_cities.csv")
summary_df.to_csv(summary_csv, index=False)

x = np.arange(len(summary_df))
width = 0.38

# --- Original overall errors figure (with Hybrid) ---
plt.figure(figsize=(9, 5), dpi=140)
plt.bar(x - width/2, summary_df["RMSE_mean"].values, width, label="RMSE")
plt.bar(x + width/2, summary_df["MAE_mean"].values,  width, label="MAE")
plt.xticks(x, summary_df["Curve"].tolist(), rotation=15)
plt.ylabel("Error (normalized)")
plt.title("Average errors across cities (all models)")
plt.grid(axis='y', alpha=0.2)
plt.legend()
plt.tight_layout()
summary_fig = os.path.join(results_dir, "errors_summary_across_cities_all_models.png")
plt.savefig(summary_fig)
plt.close()

# --- New overall errors figure WITHOUT Hybrid ---
summary_nohyb_df = summary_df[summary_df["Curve"] != "Hybrid"].reset_index(drop=True)
x_nohyb = np.arange(len(summary_nohyb_df))

plt.figure(figsize=(9, 5), dpi=140)
plt.bar(x_nohyb - width/2, summary_nohyb_df["RMSE_mean"].values, width, label="RMSE")
plt.bar(x_nohyb + width/2, summary_nohyb_df["MAE_mean"].values,  width, label="MAE")
plt.xticks(x_nohyb, summary_nohyb_df["Curve"].tolist(), rotation=15)
plt.ylabel("Error (normalized)")
plt.title("Average errors across cities (without Hybrid)")
plt.grid(axis='y', alpha=0.2)
plt.legend()
plt.tight_layout()
summary_nohyb_fig = os.path.join(results_dir, "errors_summary_across_cities_no_hybrid.png")
plt.savefig(summary_nohyb_fig)
plt.close()

# ----------------------------
# 6) Bin-based errors vs K/Kc
# ----------------------------
bin_rmse_mean = {}
bin_mae_mean  = {}

for curve in curve_names:
    rmse_means = []
    mae_means  = []
    for b in range(NBINS):
        vals_rmse = bin_rmse_lists[curve][b]
        vals_mae  = bin_mae_lists[curve][b]
        rmse_means.append(np.mean(vals_rmse) if len(vals_rmse) > 0 else np.nan)
        mae_means.append(np.mean(vals_mae)   if len(vals_mae)  > 0 else np.nan)
    bin_rmse_mean[curve] = np.array(rmse_means)
    bin_mae_mean[curve]  = np.array(mae_means)

# RMSE vs K/Kc
plt.figure(figsize=(9, 5), dpi=140)
for curve in curve_names:
    plt.plot(BIN_CENTERS, bin_rmse_mean[curve], marker="o", label=curve)
plt.xlabel("K / Kc (bin center)")
plt.ylabel("RMSE (normalized)")
plt.title(f"Bin-based average RMSE vs K/Kc (NBINS={NBINS})")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
rmse_vsK_fig = os.path.join(results_dir, "errors_rmse_vs_Kratio_bins.png")
plt.savefig(rmse_vsK_fig)
plt.close()

# MAE vs K/Kc
plt.figure(figsize=(9, 5), dpi=140)
for curve in curve_names:
    plt.plot(BIN_CENTERS, bin_mae_mean[curve], marker="o", label=curve)
plt.xlabel("K / Kc (bin center)")
plt.ylabel("MAE (normalized)")
plt.title(f"Bin-based average MAE vs K/Kc (NBINS={NBINS})")
plt.grid(alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
mae_vsK_fig = os.path.join(results_dir, "errors_mae_vs_Kratio_bins.png")
plt.savefig(mae_vsK_fig)
plt.close()

# Save bin-based errors to CSV
bin_records = []
for curve in curve_names:
    for b in range(NBINS):
        bin_records.append({
            "Curve": curve,
            "Bin_index": b,
            "K_ratio_lo": BIN_EDGES[b],
            "K_ratio_hi": BIN_EDGES[b+1],
            "K_ratio_center": BIN_CENTERS[b],
            "RMSE_bin_mean": bin_rmse_mean[curve][b],
            "MAE_bin_mean":  bin_mae_mean[curve][b],
        })
bin_errors_df = pd.DataFrame(bin_records)
bin_errors_csv = os.path.join(results_dir, "errors_by_Kratio_bins_across_cities.csv")
bin_errors_df.to_csv(bin_errors_csv, index=False)

# ----------------------------
# 7) K-weighted summary
# ----------------------------
summary_bins_rows = []
for curve in curve_names:
    rmse_curve_bins = bin_rmse_mean[curve]
    mae_curve_bins  = bin_mae_mean[curve]

    rmse_kw = float(np.nanmean(rmse_curve_bins)) if np.any(~np.isnan(rmse_curve_bins)) else np.nan
    mae_kw  = float(np.nanmean(mae_curve_bins))  if np.any(~np.isnan(mae_curve_bins))  else np.nan

    summary_bins_rows.append({
        "Curve": curve,
        "RMSE_Kweighted": rmse_kw,
        "MAE_Kweighted":  mae_kw,
    })

summary_bins_df = (
    pd.DataFrame(summary_bins_rows)
    .sort_values("MAE_Kweighted", ascending=True)
    .reset_index(drop=True)
)

summary_bins_csv = os.path.join(results_dir, "errors_Kweighted_summary_across_cities.csv")
summary_bins_df.to_csv(summary_bins_csv, index=False)

x_kw = np.arange(len(summary_bins_df))

plt.figure(figsize=(9, 5), dpi=140)
plt.bar(x_kw - width/2, summary_bins_df["RMSE_Kweighted"].values, width, label="RMSE (K-weighted)")
plt.bar(x_kw + width/2, summary_bins_df["MAE_Kweighted"].values,  width, label="MAE (K-weighted)")
plt.xticks(x_kw, summary_bins_df["Curve"].tolist(), rotation=15)
plt.ylabel("Error (normalized)")
plt.title(f"K-weighted average errors across cities (NBINS={NBINS})")
plt.grid(axis='y', alpha=0.2)
plt.legend()
plt.tight_layout()
summary_Kweighted_fig = os.path.join(results_dir, "errors_Kweighted_summary_across_cities.png")
plt.savefig(summary_Kweighted_fig)
plt.close()

print("Done.")
print("Per-city figures (with Hybrid):", figs_dir)
print("Per-city figures (no Hybrid):", figs_dir_nohyb)
print("Per-city errors CSV:", errors_csv)
print("Across-cities summary CSV (point-weighted):", summary_csv)
print("Across-cities summary figure (all models):", summary_fig)
print("Across-cities summary figure (no Hybrid):", summary_nohyb_fig)
print("Bin-based errors CSV:", bin_errors_csv)
print("RMSE vs K/Kc (bins) figure:", rmse_vsK_fig)
print("MAE vs K/Kc (bins) figure:", mae_vsK_fig)
print("K-weighted summary CSV:", summary_bins_csv)
print("K-weighted summary figure:", summary_Kweighted_fig)